# Práctica: Distribuciones de Probabilidad

En este cuaderno exploramos de forma práctica las distribuciones vistas en la teoría: visualizamos sus formas, calculamos probabilidades concretas y comprobamos empíricamente el Teorema Central del Límite mediante simulación.

In [ ]:
import sys
sys.path.append("../../src")

import numpy as np
import matplotlib.pyplot as plt
from stats_toolkit import distributions as dist

## 1. Distribución Binomial

Simulamos 50 clientes con probabilidad de conversión $p = 0.3$ y comparamos la pmf teórica con un histograma de 10.000 simulaciones.

In [ ]:
n, p = 50, 0.3
ks = np.arange(0, n + 1)
pmf_theoretical = [dist.binomial_pmf(k, n, p) for k in ks]

rng = np.random.default_rng(0)
samples = rng.binomial(n, p, size=10000)

plt.figure(figsize=(8, 4))
plt.hist(samples, bins=range(0, n + 2), density=True, alpha=0.6, label="Simulación (10000 ensayos)")
plt.plot(ks, pmf_theoretical, 'o-', color='crimson', label="pmf teórica")
plt.title(f"Binomial(n={n}, p={p})")
plt.xlabel("k (número de éxitos)")
plt.ylabel("Probabilidad")
plt.legend()
plt.show()

## 2. Distribución de Poisson

Comparamos la pmf de Poisson para distintos valores de $\lambda$: fíjate en cómo la forma se vuelve más simétrica (más "normal") a medida que $\lambda$ crece.

In [ ]:
lambdas = [1, 4, 10]
ks = np.arange(0, 25)

plt.figure(figsize=(8, 4))
for lam in lambdas:
    pmf_vals = [dist.poisson_pmf(k, lam) for k in ks]
    plt.plot(ks, pmf_vals, 'o-', label=f"λ = {lam}")

plt.title("Distribución de Poisson para distintos λ")
plt.xlabel("k")
plt.ylabel("P(X = k)")
plt.legend()
plt.show()

## 3. Distribución Normal

Superponemos la pdf teórica sobre un histograma de muestras simuladas, y calculamos una probabilidad acumulada concreta con `normal_cdf`.

In [ ]:
mu, sigma = 0, 1
samples = rng.normal(mu, sigma, size=5000)
xs = np.linspace(-4, 4, 200)
pdf_vals = [dist.normal_pdf(x, mu, sigma) for x in xs]

plt.figure(figsize=(8, 4))
plt.hist(samples, bins=50, density=True, alpha=0.6, label="Muestras simuladas")
plt.plot(xs, pdf_vals, color='crimson', label="pdf teórica")
plt.title(f"Normal(μ={mu}, σ={sigma})")
plt.xlabel("x")
plt.ylabel("Densidad")
plt.legend()
plt.show()

p = dist.normal_cdf(x=1.96, mu=0, sigma=1)
print(f"P(X <= 1.96) = {p:.4f}  (el conocido '97.5%' de la Normal estándar)")

## 4. Distribución Exponencial

Simulamos el tiempo entre tickets de soporte consecutivos, con una tasa de llegada $\lambda = 4$ tickets/hora.

In [ ]:
lam = 4
wait_times = rng.exponential(scale=1 / lam, size=5000)
xs = np.linspace(0, 2, 200)
pdf_vals = [dist.exponential_pdf(x, lam) for x in xs]

plt.figure(figsize=(8, 4))
plt.hist(wait_times, bins=50, density=True, alpha=0.6, label="Tiempos de espera simulados")
plt.plot(xs, pdf_vals, color='crimson', label="pdf teórica")
plt.title(f"Exponencial(λ={lam}) — tiempo entre tickets (horas)")
plt.xlabel("Horas")
plt.ylabel("Densidad")
plt.legend()
plt.show()

print(f"Tiempo medio de espera esperado: {1/lam:.3f} horas ({60/lam:.1f} minutos)")

## 5. Comprobando el Teorema Central del Límite

Partimos de una población muy asimétrica (Exponencial) y observamos cómo la distribución de las medias muestrales se acerca a una Normal a medida que crece el tamaño de muestra $n$.

In [ ]:
sample_sizes = [5, 30, 100]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, n in zip(axes, sample_sizes):
    means = dist.simulate_clt(
        population="exponential",
        population_params={"lam": 0.5},
        sample_size=n,
        n_samples=3000,
        seed=42,
    )
    ax.hist(means, bins=40, edgecolor="black")
    ax.set_title(f"n = {n}")
    ax.set_xlabel("Media muestral")

axes[0].set_ylabel("Frecuencia")
fig.suptitle("Convergencia hacia la Normal según crece el tamaño muestral (TCL)")
plt.tight_layout()
plt.show()

## Ejercicios propuestos

1. Una fábrica produce piezas con una tasa de defectos del 2%. Si se inspecciona un lote de 200 piezas, calcula la probabilidad de encontrar exactamente 5 defectuosas usando `binomial_pmf`, y compárala con la aproximación de Poisson (`poisson_pmf` con λ = n·p).
2. Los tiempos de respuesta de una API siguen una distribución Normal con μ = 120 ms y σ = 15 ms. ¿Qué proporción de peticiones tarda más de 150 ms? (usa `normal_cdf`).
3. Repite la simulación del TCL de la sección 5 pero partiendo de una población Uniforme y de una Poisson. ¿Cambia la velocidad de convergencia hacia la forma normal?